# Snowflake PIPE (Snowpipe) — Exact Notes

## What is PIPE

- PIPE is a named object in Snowflake
- Contains a COPY INTO statement
- Used by Snowpipe for continuous data ingestion

---

## Flow

Data Files

→ Stage

→ InsertFiles/Trigger

→ REST Endpoint

→ Ingest Queue

→ PIPE

→ Table

---

## Components

### 1. Data Files
- Incoming files (CSV, JSON, Parquet)
- Generated continuously from applications, logs, streaming systems

---

### 2. Stage
- Internal Stage (Snowflake)
- External Stage (S3, Azure, GCS)
- Acts as landing area for files

Example:
CREATE STAGE my_stage

URL='s3://mybucket/data/';

---

### 3. InsertFiles (Trigger)
- Notification that new files are available
- Trigger methods:
  - REST API
  - Auto-ingest (cloud events)

---

### 4. REST Endpoint
- API used to notify Snowflake manually

Example:
POST /v1/data/pipes/mypipe/insertFiles

---

### 5. Ingest Queue
- Internal Snowflake queue
- Stores file metadata
- Handles retries and reliability

---

### 6. PIPE

- Contains COPY INTO logic
- Executes automatically

Example:

CREATE PIPE mypipe AS

COPY INTO sales

FROM @my_stage

FILE_FORMAT = (TYPE='JSON');

---

### 7. Snowflake Compute (Serverless)

- Snowflake-managed compute
- No warehouse required
- Auto scaling

---

### 8. Target Table

Example:

CREATE TABLE sales (

  id INT,

  name STRING,

  amount NUMBER

);

---

## Key Features

- Continuous loading
- Near real-time ingestion
- Serverless
- Pause / Resume supported

---

## Pause / Resume

ALTER PIPE mypipe SET PIPE_EXECUTION_PAUSED = TRUE;

ALTER PIPE mypipe SET PIPE_EXECUTION_PAUSED = FALSE;

---

## Monitoring

SELECT * FROM TABLE(INFORMATION_SCHEMA.COPY_HISTORY());

---

## Best Practices

- File size: 10MB – 100MB
- Use compressed files
- Avoid too small or too large files

---

## Summary

- PIPE = COPY logic
- Snowpipe = ingestion engine
- Ingest Queue = buffering layer
- REST/API/Event = trigger

---

## One Line

PIPE = automatic data loading mechanism in Snowflake


# Snowflake Snowpipe + Auto Loader (End-to-End Notes)

## Overview
- Snowpipe: Continuous data ingestion in Snowflake
- Auto Loader: Databricks incremental ingestion tool
- PIPE: Object containing COPY INTO logic

---

## Step 1: Data Generation
- Sources: Logs, Kafka, IoT, CDC
- Format: JSON, CSV, Parquet

---

## Step 2: Data in Cloud Storage
- AWS S3: s3://bucket/data/
- Azure Blob
- GCS

---

## Step 3: Databricks Auto Loader (Optional)

```python
df = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "json") \
    .load("s3://bucket/data/")

df.writeStream \
    .option("checkpointLocation", "/chk") \
    .start("/output/")
```

---

## Step 4: Create Stage

```sql
CREATE STAGE my_stage
URL='s3://bucket/data/'
FILE_FORMAT=(TYPE=JSON);
```

---

## Step 5: Create Table

```sql
CREATE TABLE sales (
  id INT,
  name STRING,
  amount NUMBER
);
```

---

## Step 6: Create PIPE

```sql
CREATE PIPE mypipe AS
COPY INTO sales
FROM @my_stage;
```

---

## Step 7: Auto-Ingest

```sql
CREATE PIPE mypipe
AUTO_INGEST=TRUE
AS COPY INTO sales FROM @my_stage;
```

---

## Step 8: Flow

S3 → Stage → Ingest Queue → PIPE → Table

---

## Monitoring

### Show Pipes
```sql
SHOW PIPES;
```

### Pipe Status
```sql
SELECT SYSTEM$PIPE_STATUS('mypipe');
```

### Load History
```sql
SELECT * FROM TABLE(INFORMATION_SCHEMA.COPY_HISTORY(
TABLE_NAME=>'SALES',
START_TIME=>DATEADD(HOUR,-1,CURRENT_TIMESTAMP())
));
```

---

## Pause / Resume

```sql
ALTER PIPE mypipe SET PIPE_EXECUTION_PAUSED=TRUE;
ALTER PIPE mypipe SET PIPE_EXECUTION_PAUSED=FALSE;
```

---

## Best Practices
- File size: 10MB–100MB
- Use compression
- Avoid small files

---

## Snowpipe vs Auto Loader

| Feature | Snowpipe | Auto Loader |
|--------|----------|------------|
| Type | Ingestion | Streaming |
| Compute | Serverless | Spark |
| Use | Load to Snowflake | ETL |

---

## One Line

Snowpipe loads files automatically from cloud storage into Snowflake in near real-time.
